In [1]:
import ROOT as R
import time
import numpy as np
# Turn jsroot off if you want to make a pdf from this file.
%jsroot on
#R.EnableImplicitMT()

In [2]:
ch = R.TChain("events")
ch.Add("target_mono_filter.root")
print(f"Loaded {ch.GetEntries()} events from files.")
df = R.RDataFrame(ch)
dfx = df.Define("exit_thty","vector<double> out; for(int i=0; i<exit_py.size();++i){out.push_back(atan2(exit_py[i],exit_pz[i]));}; return out;")\
    .Define("exit_thtx","vector<double> out; for(int i=0; i<exit_px.size();++i){out.push_back(atan2(exit_px[i],exit_pz[i]));}; return out;")\
    .Define("exit_electron","vector<int> out; for(int i=0; i<exit_pdg.size();++i){if(exit_pdg[i]==11)out.push_back(i);}; return out;")\
    .Define("exit_positron","vector<int> out; for(int i=0; i<exit_pdg.size();++i){if(exit_pdg[i]==-11)out.push_back(i);}; return out;")\
    .Define("exit_photon","vector<int> out; for(int i=0; i<exit_pdg.size();++i){if(exit_pdg[i]==22)out.push_back(i);}; return out;")

Loaded 12668 events from files.


In [3]:
print(f"Available data names in Tuple:")
All_Names = list(dfx.GetColumnNames())
ii=0
for n in All_Names:
    s = str(n)
    dat_type = dfx.GetColumnType(s).replace("ROOT::VecOps::RVec<","RVec<")
    print(f"{s:<30s} [{dat_type:<20s}]", end = " ")
    ii += 1
    if ii%2==0:     # Replace the 2 with 3 to get 3 columns (too wide for my screen)
        print("")


Available data names in Tuple:
Edep                           [Double_t            ] created_E                      [RVec<double>        ] 
created_parent                 [RVec<int>           ] created_particle               [RVec<Char_t>        ] 
created_pdg                    [RVec<int>           ] created_primary                [RVec<int>           ] 
created_process                [RVec<Char_t>        ] created_px                     [RVec<double>        ] 
created_py                     [RVec<double>        ] created_pz                     [RVec<double>        ] 
created_track                  [RVec<int>           ] created_vx                     [RVec<double>        ] 
created_vy                     [RVec<double>        ] created_vz                     [RVec<double>        ] 
created_x                      [RVec<double>        ] created_y                      [RVec<double>        ] 
created_z                      [RVec<double>        ] event                          [Int_t      

In [5]:
dfxx=dfx.Define("nele","exit_electron.size()").Define("ngamma","exit_photon.size()").Define("npos","exit_positron.size()")
pr = dfxx.Display(("event","exit_E","exit_px","exit_py","exit_pz","exit_pdg","nPrimary","nCreated","nExit","nele","ngamma"), 30)
pr

Row,event,exit_E,exit_px,exit_py,exit_pz,exit_pdg,nPrimary,nCreated,nExit,nele,ngamma
0,2212,3710.386485,42.720960,-85.983003,3709.655202,11,1498,143,1,1,0
1,2217,8.910100,0.204386,0.131132,8.906790,22,1498,148,3,0,3
,,0.075436,-0.022186,0.004818,0.071938,22,,,,,
,,0.059483,-0.022095,0.001521,0.055206,22,,,,,
2,2219,11.081528,0.237524,-0.246362,11.076242,22,1498,137,1,0,1
3,2220,0.039672,0.001223,-0.002532,0.039572,22,1498,132,1,0,1
4,2238,0.009692,0.003352,-0.000199,0.009092,22,1498,145,1,0,1
5,2242,12.195096,0.506302,0.205146,12.182855,22,1498,146,1,0,1
6,2249,0.031329,0.008456,-0.000690,0.030159,22,1498,120,1,0,1
7,2270,0.091603,0.059510,-0.005803,0.069397,22,1498,140,1,0,1


In [7]:
h_n_all = dfxx.Histo1D(("h_n_all","Number of Particles",11,-0.5,10.5),"nExit")
h_n_e = dfxx.Histo1D(("h_n_e","Number of e-",11,-0.5,10.5),"nele")
h_n_g = dfxx.Histo1D(("h_n_e","Number of gamma",11,-0.5,10.5),"ngamma")
h_n_p = dfxx.Histo1D(("h_n_e","Number of e+",11,-0.5,10.5),"npos")

cc1 = R.TCanvas("cc1","cc1", 800,600)
cc1.SetLogy()
h_n_all.Draw()
h_n_e.SetLineColor(R.kRed)
h_n_e.Draw("same")
h_n_g.SetLineColor(R.kGreen)
h_n_g.Draw("same")
h_n_p.SetLineColor(R.kViolet)
h_n_p.Draw("same")
cc1.Draw()

<IPython.core.display.Javascript object>

In [8]:
h_exit_thty_all = dfx.Histo1D(("h_exit_thty_all","Theta_y all",1000,-0.11,0.11 ),"exit_thty")
h_exit_thty_e = dfx.Define("thty","vector<double> out; for(int i : exit_electron) out.push_back(exit_thty[i]); return out;").Histo1D(("h_exit_thty_e","Theta_y e-",1000,-0.11,0.11 ),"thty")
h_exit_thty_g = dfx.Define("thty","vector<double> out; for(int i : exit_photon) out.push_back(exit_thty[i]); return out;").Histo1D(("h_exit_thty_e","Theta_y gamma",1000,-0.11,0.11 ),"thty")
h_exit_thty_p = dfx.Define("thty","vector<double> out; for(int i : exit_positron) out.push_back(exit_thty[i]); return out;").Histo1D(("h_exit_thty_p","Theta_y e+",1000,-0.11,0.11 ),"thty")
cc1 = R.TCanvas("cc1","cc1", 800,600)
cc1.SetLogy()
h_exit_thty_all.Draw()
h_exit_thty_e.SetLineColor(R.kRed)
h_exit_thty_e.Draw("same")
h_exit_thty_g.SetLineColor(R.kGreen)
h_exit_thty_g.Draw("same")
h_exit_thty_p.SetLineColor(R.kViolet)
h_exit_thty_p.Draw("same")
cc1.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: cc1


<IPython.core.display.Javascript object>

In [9]:
h_exit_E_all = dfx.Histo1D(("h_exit_E_all","E all",1000,-0., 4000. ),"exit_E")
h_exit_E_e = dfx.Define("E","vector<double> out; for(int i : exit_electron) out.push_back(exit_E[i]); return out;").Histo1D(("h_exit_E_e","E e-",1000,-0.,4000. ),"E")
h_exit_E_g = dfx.Define("E","vector<double> out; for(int i : exit_photon) out.push_back(exit_E[i]); return out;").Histo1D(("h_exit_E_e","E gamma",1000,-0.,4000. ),"E")
h_exit_E_p = dfx.Define("E","vector<double> out; for(int i : exit_positron) out.push_back(exit_E[i]); return out;").Histo1D(("h_exit_E_p","E e+",1000,-0.,4000. ),"E")
cc2 = R.TCanvas("cc2","cc2", 800,600)
cc2.SetLogy()
h_exit_E_all.Draw()
h_exit_E_e.SetLineColor(R.kRed)
h_exit_E_e.Draw("same")
h_exit_E_g.SetLineColor(R.kGreen)
h_exit_E_g.Draw("same")
h_exit_E_p.SetLineColor(R.kViolet)
h_exit_E_p.Draw("same")
cc2.Draw()

<IPython.core.display.Javascript object>